In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz, save_npz
import pickle

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold


# Metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")

✅ All libraries imported successfully!
XGBoost version: 3.1.1


In [ ]:
from google.colab import files
print("Please upload your CSV file (phishing_features_10k.csv)...")
uploaded = files.upload()]

print("File uploaded successfully!")

📤 Please upload your CSV file (phishing_features_10k.csv)...


Saving phishing_features_10k.csv to phishing_features_10k.csv
✅ File uploaded successfully!


In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("=" * 60)
print("DATASET ANALYSIS")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"Class balance: {df['label'].value_counts(normalize=True)}")

📊 DATASET ANALYSIS
Shape: (10000, 5001)
Label distribution:
label
0    5000
1    5000
Name: count, dtype: int64
Class balance: label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('label', axis=1)
y = df['label']

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("=" * 60)
print("REALISTIC DATA SPLIT (80-20)")
print("=" * 60)
print(f"Training: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test:     {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")


📊 REALISTIC DATA SPLIT (80-20)
Training: 8,000 samples (80.0%)
Test:     2,000 samples (20.0%)


In [ ]:
print("=" * 60)
print("CLASS IMBALANCE HANDLING")
print("=" * 60)

# Check if imbalance exists
class_counts = pd.Series(y_train).value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Imbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 1.5:
    print("Applying SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
    print(f"After SMOTE: {pd.Series(y_train_balanced).value_counts()}")
else:
    print("Dataset is balanced, no resampling needed.")
    X_train_balanced = X_train_selected
    y_train_balanced = y_train

⚖️ CLASS IMBALANCE HANDLING
Imbalance ratio: 1.00
Dataset is balanced, no resampling needed.


In [ ]:
def cross_validate_model(model, X, y, cv=5):
    """Perform stratified k-fold cross-validation"""
    print(f"\nPerforming {cv}-Fold Cross-Validation...")

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    # Calculate multiple metrics
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    cv_results = {}

    for metric in scoring:
        scores = cross_val_score(model, X, y, cv=skf, scoring=metric, n_jobs=-1)
        cv_results[metric] = {
            'mean': scores.mean(),
            'std': scores.std(),
            'scores': scores
        }
        print(f"{metric:10s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

    return cv_results

In [ ]:
from sklearn.model_selection import StratifiedKFold


print("RANDOM FOREST WITH HYPERPARAMETER TUNING")


# Define parameter grid
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Initialize model
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

# Grid search with cross-validation
print("\nPerforming Grid Search (this may take a while)...")
rf_grid_search = GridSearchCV(
    rf_base,
    rf_param_grid,
    cv=3,  # 3-fold CV for faster results
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

rf_grid_search.fit(X_train_balanced, y_train_balanced)


print(f"\nBest CV F1-score: {rf_grid_search.best_score_:.4f}")

# Use best model
rf_model = rf_grid_search.best_estimator_

# Cross-validate best model
rf_cv_results = cross_validate_model(rf_model, X_train_balanced, y_train_balanced)


🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲
RANDOM FOREST WITH HYPERPARAMETER TUNING
🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲

⏳ Performing Grid Search (this may take a while)...
Fitting 3 folds for each of 216 candidates, totalling 648 fits

Best CV F1-score: 0.9161

🔄 Performing 5-Fold Cross-Validation...
accuracy  : 0.9133 (+/- 0.0038)
precision : 0.8713 (+/- 0.0084)
recall    : 0.9703 (+/- 0.0072)
f1        : 0.9182 (+/- 0.0031)
roc_auc   : 0.9664 (+/- 0.0023)


In [ ]:
from sklearn.model_selection import StratifiedKFold

print("XGBOOST WITH HYPERPARAMETER TUNING")


# Define parameter grid
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Initialize model
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)

# Grid search
print("\n⏳ Performing Grid Search...")
xgb_grid_search = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

xgb_grid_search.fit(X_train_balanced, y_train_balanced)

print("\n✅ Best parameters found:")
print(xgb_grid_search.best_params_)
print(f"\nBest CV F1-score: {xgb_grid_search.best_score_:.4f}")

# Use best model
xgb_model = xgb_grid_search.best_estimator_

# Cross-validate
xgb_cv_results = cross_validate_model(xgb_model, X_train_balanced, y_train_balanced)

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
XGBOOST WITH HYPERPARAMETER TUNING
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

⏳ Performing Grid Search...
Fitting 3 folds for each of 108 candidates, totalling 324 fits

✅ Best parameters found:
{'colsample_bytree': 1.0, 'learning_rate': 0.3, 'max_depth': 9, 'n_estimators': 40, 'subsample': 0.8}

Best CV F1-score: 0.9216

🔄 Performing 5-Fold Cross-Validation...
accuracy  : 0.9155 (+/- 0.0042)
precision : 0.8698 (+/- 0.0084)
recall    : 0.9775 (+/- 0.0077)
f1        : 0.9204 (+/- 0.0037)
roc_auc   : 0.9732 (+/- 0.0013)


In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression


print("HYBRID ENSEMBLE MODELS")


# ============================================================
# APPLY FEATURE SELECTION TO TEST SET
# ============================================================
print(f"\n  Applying feature selection to test set...")
print(f"Original test shape: {X_test.shape}")
X_test_selected = selector.transform(X_test)
print(f"Transformed test shape: {X_test_selected.shape}")

# ============================================================
# 1. SOFT VOTING
# ============================================================

print(" SOFT VOTING ENSEMBLE")


voting_soft = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft',
    n_jobs=-1
)
voting_soft.fit(X_train_selected, y_train_balanced)
voting_soft_cv = cross_validate_model(voting_soft, X_train_selected, y_train_balanced)

# ============================================================
# 2. HARD VOTING (NO PROBABILITIES)
# ============================================================

print("HARD VOTING ENSEMBLE")


voting_hard = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='hard'
)
voting_hard.fit(X_train_selected, y_train_balanced)
voting_hard_cv = cross_validate_model(voting_hard, X_train_selected, y_train_balanced)

# ============================================================
# 3. WEIGHTED VOTING
# ============================================================

print("WEIGHTED VOTING")


weights = [rf_cv_results['f1']['mean'], xgb_cv_results['f1']['mean']]
weighted_voting = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft',
    weights=weights
)
weighted_voting.fit(X_train_selected, y_train_balanced)
weighted_cv = cross_validate_model(weighted_voting, X_train_selected, y_train_balanced)

# ============================================================
# 4. STACKING ENSEMBLE
# ============================================================

print("STACKING ENSEMBLE")


stacking = StackingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train_selected, y_train_balanced)
stacking_cv = cross_validate_model(stacking, X_train_selected, y_train_balanced)

# ============================================================
# FINAL SUMMARY TABLE (CV-BASED ON ACCURACY)
# ============================================================

print("MODEL COMPARISON BASED ON 5-FOLD CROSS-VALIDATION (ACCURACY)")


models_cv_accuracy = {
    'Random Forest': rf_cv_results['accuracy']['mean'],
    'XGBoost': xgb_cv_results['accuracy']['mean'],
    'Soft Voting': voting_soft_cv['accuracy']['mean'],
    'Hard Voting': voting_hard_cv['accuracy']['mean'],
    'Weighted Voting': weighted_cv['accuracy']['mean'],
    'Stacking': stacking_cv['accuracy']['mean'],
}

for model_name, acc in models_cv_accuracy.items():
    print(f"{model_name:<20} mean accuracy: {acc:.4f}")

best_model = max(models_cv_accuracy, key=models_cv_accuracy.get)
print(f"\n Best Model (by accuracy): {best_model}")
print("\n ENSEMBLE ANALYSIS COMPLETE")


🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗
HYBRID ENSEMBLE MODELS
🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗

⚠️  Applying feature selection to test set...
Original test shape: (2000, 5000)
Transformed test shape: (2000, 5)

1️⃣  SOFT VOTING ENSEMBLE

🔄 Performing 5-Fold Cross-Validation...
accuracy  : 0.9168 (+/- 0.0037)
precision : 0.8716 (+/- 0.0081)
recall    : 0.9778 (+/- 0.0083)
f1        : 0.9215 (+/- 0.0032)
roc_auc   : 0.9742 (+/- 0.0012)

2️⃣  HARD VOTING ENSEMBLE

🔄 Performing 5-Fold Cross-Validation...
accuracy  : 0.9160 (+/- 0.0033)
precision : 0.8781 (+/- 0.0066)
recall    : 0.9662 (+/- 0.0066)
f1        : 0.9200 (+/- 0.0030)
roc_auc   : nan (+/- nan)

3️⃣  WEIGHTED VOTING

🔄 Performing 5-Fold Cross-Validation...
accuracy  : 0.9168 (+/- 0.0037)
precision : 0.8716 (+/- 0.0081)
recall    : 0.9778 (+/- 0.0083)
f1        : 0.9215 (+/- 0.0032)
roc_auc   : 0.9742 (+/- 0.0012)

4️⃣  STACKING ENSEMBLE

🔄 Performing 5-Fold Cross-Validation...
accuracy  : 0.9166 (+/- 0.0039)
precision : 0.8725 (+/- 0.0068

In [ ]:
import joblib

# Save the best model
best_model_name = best_model  # from your previous code
print(f" Saving the best model: {best_model_name}")

# Map model names to actual trained objects
model_map = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Soft Voting': voting_soft,
    'Hard Voting': voting_hard,
    'Weighted Voting': weighted_voting,
    'Stacking': stacking
}

# Get the best model object
best_model_obj = model_map[best_model_name]

# Save it to a file
model_filename = f"{best_model_name.replace(' ', '_').lower()}_best_model.pkl"
joblib.dump(best_model_obj, model_filename)

print(f" Model saved as '{model_filename}'")


💾 Saving the best model: Soft Voting
✅ Model saved as 'soft_voting_best_model.pkl'


In [ ]:
import joblib
from google.colab import files

# Save the best model
best_model_obj = model_map[best_model]  # from previous mapping
model_filename = f"{best_model.replace(' ', '_').lower()}_best_model.pkl"
joblib.dump(best_model_obj, model_filename)

# Download the file
files.download(model_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>